# Chapter 6: Vector Search with Lucene

Lucene stores vectors as one more field type of a document, next to the text and numeric
fields of demo ch04-04. This demo follows the Lucene parts of book section 6.5 with the
movie collection of demo ch06-04: the same 10,000 movies, the same Qwen3 embeddings, and the
same query.

> a mind-bending heist thriller with Leonardo DiCaprio, released after 2005, rated 7.5 or better

We build an index with a vector field, run an approximate k-NN query, add filters the right
and the wrong way, combine the vector query with BM25 clauses, read the score with
`explain`, and fuse the two rankings with RRF instead. A last section tries Lucene's
quantized vector formats.

<div style="border-left: 4px solid #D97706; background: rgba(217, 119, 6, 0.06); padding: 0.6em 0.9em; margin: 0.6em 0; border-radius: 4px;">
<strong style="color:#D97706; text-transform:uppercase; font-size:0.78em; letter-spacing:0.06em;">Requires the Ganymede Java kernel</strong><br>
Like ch04-04, this demo runs on a Java kernel, not Python, because it uses real Apache Lucene. Install the Ganymede Java 21 kernel first: see the "Java kernel" section of the <a href="README.md">README.md</a> in this folder. Then select <code>Ganymede 2.1.1 (Java 21)</code> as the notebook kernel.
</div>

**Prerequisites:** ch04-04 (Lucene fields, BM25, `explain`), ch06-04 (the movie query in
FAISS, OpenSearch, and pgvector).

The first cell downloads Lucene 10 and the Jackson JSON library from Maven Central onto the
kernel classpath.

In [1]:
%%pom
dependencies:
- org.apache.lucene:lucene-core:10.2.1
- org.apache.lucene:lucene-analysis-common:10.2.1
- org.apache.lucene:lucene-codecs:10.2.1
- com.fasterxml.jackson.core:jackson-databind:2.18.2

## Loading the movie collection

The collection ships with demo ch06-04 in `eval/ch06_movies/`: one JSON line per movie, and
the embeddings as a NumPy archive of float16 matrices. A NumPy archive is a ZIP file of
`.npy` files, each a short text header followed by the raw numbers, so a few lines of Java
read it.

In [2]:
import com.fasterxml.jackson.databind.ObjectMapper;
import java.io.*;
import java.nio.*;
import java.nio.file.*;
import java.util.*;
import java.util.regex.*;
import java.util.zip.ZipFile;

record Movie(int id, String title, int year, float rating, List<String> genres,
             List<String> actors, String overview) {}

Path DATA = Path.of("eval", "ch06_movies");
ObjectMapper json = new ObjectMapper();
List<Movie> movies = new ArrayList<>();
for (String line : Files.readAllLines(DATA.resolve("movies.jsonl"))) {
    var m = json.readTree(line);
    movies.add(new Movie(m.get("id").asInt(), m.get("title").asText(), m.get("year").asInt(),
        (float) m.get("rating").asDouble(),
        json.convertValue(m.get("genres"), List.class), json.convertValue(m.get("actors"), List.class),
        m.get("overview").asText()));
}

// Read one .npy entry of the archive: header {'descr': ..., 'shape': (rows, cols)}, then data
ByteBuffer npyEntry(ZipFile zip, String name, String[] header) throws IOException {
    byte[] raw = zip.getInputStream(zip.getEntry(name + ".npy")).readAllBytes();
    ByteBuffer buf = ByteBuffer.wrap(raw).order(ByteOrder.LITTLE_ENDIAN);
    int headerLen = raw[6] == 1 ? Short.toUnsignedInt(buf.getShort(8)) : buf.getInt(8);
    int start = raw[6] == 1 ? 10 : 12;
    header[0] = new String(raw, start, headerLen);
    return buf.position(start + headerLen).slice().order(ByteOrder.LITTLE_ENDIAN);
}

// float16 matrix -> unit-length float32 rows (unit length: dot product = cosine)
float[][] readMatrix(ZipFile zip, String name) throws IOException {
    String[] header = new String[1];
    ByteBuffer buf = npyEntry(zip, name, header);
    Matcher shape = Pattern.compile("\\((\\d+), (\\d+)\\)").matcher(header[0]);
    shape.find();
    float[][] rows = new float[Integer.parseInt(shape.group(1))][Integer.parseInt(shape.group(2))];
    for (float[] row : rows) {
        double norm = 0;
        for (int j = 0; j < row.length; j++) {
            row[j] = Float.float16ToFloat(buf.getShort());
            norm += row[j] * row[j];
        }
        for (int j = 0; j < row.length; j++) row[j] /= (float) Math.sqrt(norm);
    }
    return rows;
}

// Array of fixed-width unicode strings ('<U53'): UTF-32 code points, padded with zeros
List<String> readStrings(ZipFile zip, String name) throws IOException {
    String[] header = new String[1];
    ByteBuffer buf = npyEntry(zip, name, header);
    Matcher m = Pattern.compile("<U(\\d+)'.*\\((\\d+),\\)").matcher(header[0]);
    m.find();
    int width = Integer.parseInt(m.group(1));
    List<String> out = new ArrayList<>();
    for (int i = 0; i < Integer.parseInt(m.group(2)); i++) {
        StringBuilder sb = new StringBuilder();
        for (int c = 0; c < width; c++) {
            int cp = buf.getInt();
            if (cp != 0) sb.appendCodePoint(cp);
        }
        out.add(sb.toString());
    }
    return out;
}

float[][] X;
Map<String, float[]> QUERIES = new LinkedHashMap<>();
try (ZipFile zip = new ZipFile(DATA.resolve("embeddings.npz").toFile())) {
    X = readMatrix(zip, "docs");
    float[][] qs = readMatrix(zip, "queries");
    List<String> texts = readStrings(zip, "query_texts");
    for (int i = 0; i < texts.size(); i++) QUERIES.put(texts.get(i), qs[i]);
}
float[] q = QUERIES.get("a mind-bending heist thriller");
Movie inception = movies.stream().filter(m -> m.title().equals("Inception")).findFirst().get();
System.out.printf("%,d movies, embeddings %,d x %d, %d query embeddings%n",
    movies.size(), X.length, X[0].length, QUERIES.size());
System.out.println("Inception: " + inception.year() + ", rating " + inception.rating()
    + ", " + inception.genres() + ", " + inception.actors());

10,000 movies, embeddings 10,000 x 1024, 5 query embeddings
Inception: 2010, rating 8.1, [Action, Thriller, Science Fiction, Mystery, Adventure], [Leonardo DiCaprio, Joseph Gordon-Levitt, Ellen Page, Tom Hardy, Cillian Murphy]


We also compute the exact answer for the vector part of the query, the cosine of the query
with every overview, as the reference for the approximate search.

In [3]:
float dot(float[] a, float[] b) {
    float s = 0;
    for (int i = 0; i < a.length; i++) s += a[i] * b[i];
    return s;
}

// Movie ids sorted by decreasing cosine to v, optionally restricted to a predicate
List<Integer> exactRanking(float[] v, java.util.function.Predicate<Movie> admit) {
    return movies.stream().filter(admit).map(Movie::id)
        .sorted(Comparator.comparingDouble((Integer i) -> -dot(X[i], v))).toList();
}

List<Integer> exact = exactRanking(q, m -> true);
String title(int id) { return movies.get(id).title() + " (" + movies.get(id).year() + ")"; }
System.out.println("Exact top 5: " + exact.subList(0, 5).stream().map(id -> title(id)).toList());
System.out.println("Inception: exact rank " + (exact.indexOf(inception.id()) + 1));

Exact top 5: [Parker (2013), Marnie (1964), Sneakers (1992), Marauders (2016), Shimmer Lake (2017)]
Inception: exact rank 138


As in ch06-04, the vector part alone ranks Inception far down.

## Vectors next to the inverted index

A movie becomes a Lucene document with text fields for the title, the actors, and the
genres, numeric fields for the year and the rating, and a `KnnFloatVectorField` for the
overview embedding. The similarity function belongs to the field.

In [4]:
import org.apache.lucene.analysis.standard.StandardAnalyzer;
import org.apache.lucene.document.*;
import org.apache.lucene.index.*;
import org.apache.lucene.search.*;
import org.apache.lucene.store.*;

Document toDocument(Movie m) {
    Document doc = new Document();
    doc.add(new StoredField("id", m.id()));
    doc.add(new TextField("title", m.title(), Field.Store.YES));
    doc.add(new TextField("actors", String.join(", ", m.actors()), Field.Store.NO));
    doc.add(new TextField("genres", String.join(", ", m.genres()), Field.Store.NO));
    doc.add(new IntField("year", m.year(), Field.Store.YES));
    doc.add(new FloatField("rating", m.rating(), Field.Store.YES));
    doc.add(new KnnFloatVectorField("embedding", X[m.id()], VectorSimilarityFunction.COSINE));
    return doc;
}

Directory buildIndex(IndexWriterConfig config) throws IOException {
    Directory dir = new ByteBuffersDirectory();
    try (IndexWriter writer = new IndexWriter(dir, config)) {
        for (Movie m : movies) writer.addDocument(toDocument(m));
    }
    return dir;
}

long bytesOnDisk(Directory dir) throws IOException {
    long total = 0;
    for (String file : dir.listAll()) total += dir.fileLength(file);
    return total;
}

long start = System.nanoTime();
Directory directory = buildIndex(new IndexWriterConfig(new StandardAnalyzer()));
double buildSeconds = (System.nanoTime() - start) / 1e9;
IndexReader reader = DirectoryReader.open(directory);
IndexSearcher searcher = new IndexSearcher(reader);
StoredFields stored = searcher.storedFields();
System.out.println(String.format("Indexed %,d movies in %.1f s: %d segments, %.0f MB",
    reader.numDocs(), buildSeconds, reader.leaves().size(), bytesOnDisk(directory) / 1e6));

Indexed 10,000 movies in 11.7 s: 3 segments, 42 MB


Oct 20, 2026 7:50:56 PM org.apache.lucene.internal.vectorization.VectorizationProvider lookup


Lucene builds an HNSW graph for the vector field of every segment. While the writer adds
documents, it flushes several segments, and each one gets its own graph. A query searches
every segment's graph and merges the results, the same scatter-gather pattern as for text
queries. Building the graphs is why indexing vectors costs much more time than indexing
text.

## Approximate k-NN query

`KnnFloatVectorQuery` names the vector field, the query vector, and $k$. We print the top
10 next to the exact cosine.

In [5]:
int movieId(int luceneDoc) throws IOException {
    return stored.document(luceneDoc).getField("id").numericValue().intValue();
}

List<Integer> ids(TopDocs top) throws IOException {
    List<Integer> out = new ArrayList<>();
    for (ScoreDoc sd : top.scoreDocs) out.add(movieId(sd.doc));
    return out;
}

void showHits(TopDocs top, int limit) throws IOException {
    System.out.printf("%-5s %-40s %-9s %-9s %s%n", "Rank", "Movie", "Score", "Cosine", "(1+cos)/2");
    for (int r = 0; r < Math.min(limit, top.scoreDocs.length); r++) {
        int id = movieId(top.scoreDocs[r].doc);
        float cos = dot(X[id], q);
        System.out.printf("%-5d %-40s %-9.4f %-9.4f %.4f%n", r + 1, title(id),
            top.scoreDocs[r].score, cos, (1 + cos) / 2);
    }
}

int overlap(List<Integer> found, List<Integer> reference, int k) {
    Set<Integer> ref = new HashSet<>(reference.subList(0, Math.min(k, reference.size())));
    return (int) found.stream().limit(k).filter(ref::contains).count();
}

TopDocs knnTop = searcher.search(new KnnFloatVectorQuery("embedding", q, 10), 10);
showHits(knnTop, 10);
System.out.println("Recall@10 against exact search: " + overlap(ids(knnTop), exact, 10) / 10.0);

Rank  Movie                                    Score     Cosine    (1+cos)/2
1     Parker (2013)                            0.7992    0.5985    0.7992
2     Sneakers (1992)                          0.7821    0.5642    0.7821
3     Marauders (2016)                         0.7799    0.5598    0.7799
4     Shimmer Lake (2017)                      0.7799    0.5598    0.7799
5     The Collector (2009)                     0.7770    0.5540    0.7770
6     Man on a Ledge (2012)                    0.7753    0.5506    0.7753
7     Lost Highway (1997)                      0.7751    0.5502    0.7751
8     Now You See Me (2013)                    0.7742    0.5484    0.7742
9     Baby Driver (2017)                       0.7741    0.5481    0.7741
10    Limitless (2011)                         0.7727    0.5453    0.7727
Recall@10 against exact search: 0.8


The score is not the cosine itself. Lucene reports $(1 + \cos)/2$ for the cosine
similarity, so that scores are positive and higher is always better, the same convention as
OpenSearch in ch06-04.

## Filters: inside the vector query, not next to it

The filters "released after 2005" and "rated 7.5 or better" are range queries on the numeric
fields. Lucene passes them as the fourth argument of `KnnFloatVectorQuery` and searches the
graph only for admitted documents. If few documents pass the filter, Lucene runs an exact
search over them instead.

In [6]:
Query filter = new BooleanQuery.Builder()
    .add(IntField.newRangeQuery("year", 2006, Integer.MAX_VALUE), BooleanClause.Occur.FILTER)
    .add(FloatField.newRangeQuery("rating", 7.5f, Float.POSITIVE_INFINITY), BooleanClause.Occur.FILTER)
    .build();
int admittedCount = searcher.count(filter);
List<Integer> exactFiltered = exactRanking(q, m -> m.year() > 2005 && m.rating() >= 7.5f);

TopDocs inside = searcher.search(new KnnFloatVectorQuery("embedding", q, 10, filter), 10);
System.out.printf("Admitted by the filter: %d of %,d movies%n%n", admittedCount, reader.numDocs());
showHits(inside, 10);
System.out.println("Recall@10 against exact filtered search: " + overlap(ids(inside), exactFiltered, 10) / 10.0);

Admitted by the filter: 261 of 10,000 movies

Rank  Movie                                    Score     Cosine    (1+cos)/2
1     River (2015)                             0.7463    0.4927    0.7463
2     Inception (2010)                         0.7411    0.4822    0.7411
3     The Lost Room (2006)                     0.7386    0.4773    0.7386
4     Kingsman: The Secret Service (2015)      0.7375    0.4750    0.7375
5     An Inspector Calls (2015)                0.7299    0.4597    0.7299
6     Seven Pounds (2008)                      0.7282    0.4564    0.7282
7     No Country for Old Men (2007)            0.7275    0.4551    0.7275
8     The Wolf of Wall Street (2013)           0.7247    0.4494    0.7247
9     The Imitation Game (2014)                0.7234    0.4467    0.7234
10    Who Am I (2014)                          0.7084    0.4168    0.7084
Recall@10 against exact filtered search: 1.0


The book warns against a tempting alternative: an unfiltered `KnnFloatVectorQuery` combined
with the filter as a `FILTER` clause of a Boolean query. The two parts then run
independently: the vector query returns its top 10 over all movies, and the Boolean query
intersects them with the filter. This is post-filtering.

In [7]:
Query postFiltered = new BooleanQuery.Builder()
    .add(new KnnFloatVectorQuery("embedding", q, 10), BooleanClause.Occur.MUST)
    .add(filter, BooleanClause.Occur.FILTER)
    .build();
TopDocs outside = searcher.search(postFiltered, 10);
System.out.println("Filter inside the vector query:  " + inside.scoreDocs.length + " results");
System.out.println("Filter next to the vector query: " + outside.scoreDocs.length + " results "
    + ids(outside).stream().map(id -> title(id)).toList());

Filter inside the vector query:  10 results
Filter next to the vector query: 0 results []


The query asks for 10 movies and silently gets far fewer, here none at all: only the movies
of the unfiltered top 10 that happen to pass the filter survive, and none of them does.

## Hybrid query: vector score plus BM25

The keyword part becomes ordinary BM25 clauses of a Boolean query: a phrase query for the
actor and a term query for the genre. The filtered vector query is one more `SHOULD`
clause, and the filter applies to the whole query.

In [8]:
import org.apache.lucene.index.Term;

Query knn50 = new KnnFloatVectorQuery("embedding", q, 50, filter);
Query actor = new PhraseQuery("actors", "leonardo", "dicaprio");
Query genre = new TermQuery(new Term("genres", "thriller"));
Query hybrid = new BooleanQuery.Builder()
    .add(knn50, BooleanClause.Occur.SHOULD)
    .add(actor, BooleanClause.Occur.SHOULD)
    .add(genre, BooleanClause.Occur.SHOULD)
    .add(filter, BooleanClause.Occur.FILTER)
    .build();
TopDocs hybridTop = searcher.search(hybrid, 10);
System.out.printf("%-5s %-40s %s%n", "Rank", "Movie", "Score");
for (int r = 0; r < hybridTop.scoreDocs.length; r++)
    System.out.printf("%-5d %-40s %.4f%n", r + 1, title(movieId(hybridTop.scoreDocs[r].doc)),
        hybridTop.scoreDocs[r].score);

Rank  Movie                                    Score
1     The Departed (2006)                      6.0139
2     Shutter Island (2010)                    6.0039
3     Before the Flood (2016)                  5.8614
4     Inception (2010)                         5.6755
5     The Wolf of Wall Street (2013)           5.4874
6     Django Unchained (2012)                  5.2288
7     Who Am I (2014)                          1.4933
8     The Thinning (2016)                      1.4610
9     I Saw the Devil (2010)                   1.3279
10    The Lives of Others (2006)               1.3270


The DiCaprio movies take the top places, ordered mostly by their BM25 scores. A Boolean
query adds the scores of its clauses, and `explain` shows how much each clause contributes.

In [9]:
int inceptionDoc = Arrays.stream(hybridTop.scoreDocs)
    .filter(sd -> { try { return movieId(sd.doc) == inception.id(); } catch (IOException e) { return false; } })
    .findFirst().get().doc;

// Print the top two levels of the explanation: the sum and its clauses
Explanation e = searcher.explain(hybrid, inceptionDoc);
System.out.printf("%.4f = %s%n", e.getValue().floatValue(), e.getDescription());
for (Explanation part : e.getDetails())
    System.out.printf("  %.4f = %s%n", part.getValue().floatValue(), part.getDescription());

5.6755 = sum of:
  0.7411 = within top 50 docs
  4.5495 = weight(actors:"leonardo dicaprio" in 0) [BM25Similarity], result of:
  0.3850 = weight(genres:thriller in 0) [BM25Similarity], result of:
  0.0000 = match on required clause, product of:
  0.0000 = match on required clause, product of:


The vector clause contributes at most 1, its $(1 + \cos)/2$ score. A BM25 clause for a rare
name like "dicaprio" contributes several points, so the keyword part dominates the sum.
Lucene itself provides no rank fusion; applications either weight the clauses with boosts or
run the two queries separately and fuse their rankings.

## Rank fusion instead of a score sum

We run the filtered vector query and the keyword query separately, take the top 50 of each,
and fuse them with reciprocal rank fusion, as in ch05-05 and ch06-04.

In [10]:
Query keywords = new BooleanQuery.Builder()
    .add(actor, BooleanClause.Occur.SHOULD)
    .add(genre, BooleanClause.Occur.SHOULD)
    .add(filter, BooleanClause.Occur.FILTER)
    .build();
List<Integer> dense = ids(searcher.search(knn50, 50));
List<Integer> sparse = ids(searcher.search(keywords, 50));

List<Integer> rrf(List<List<Integer>> rankings, int k) {
    Map<Integer, Double> scores = new HashMap<>();
    for (List<Integer> ranking : rankings)
        for (int r = 0; r < ranking.size(); r++)
            scores.merge(ranking.get(r), 1.0 / (k + r + 1), Double::sum);
    return scores.entrySet().stream()
        .sorted(Map.Entry.<Integer, Double>comparingByValue().reversed())
        .map(Map.Entry::getKey).toList();
}

List<Integer> fused = rrf(List.of(dense, sparse), 60);
List<Integer> summed = ids(hybridTop);
System.out.printf("%-5s %-40s %-7s %-7s %s%n", "Rank", "RRF", "Dense", "BM25", "Score sum (above)");
for (int r = 0; r < 10; r++) {
    int id = fused.get(r);
    System.out.printf("%-5d %-40s %-7s %-7s %s%n", r + 1, title(id),
        dense.contains(id) ? dense.indexOf(id) + 1 : "-", sparse.contains(id) ? sparse.indexOf(id) + 1 : "-",
        r < summed.size() ? title(summed.get(r)) : "");
}

Rank  RRF                                      Dense   BM25    Score sum (above)
1     Inception (2010)                         2       4       The Departed (2006)
2     The Wolf of Wall Street (2013)           8       5       Shutter Island (2010)
3     Who Am I (2014)                          10      7       Before the Flood (2016)
4     The Departed (2006)                      17      3       Inception (2010)
5     No Country for Old Men (2007)            7       18      The Wolf of Wall Street (2013)
6     Shutter Island (2010)                    32      2       Django Unchained (2012)
7     An Inspector Calls (2015)                5       29      Who Am I (2014)
8     The Raid 2 (2014)                        12      22      The Thinning (2016)
9     Gone Girl (2014)                         21      13      I Saw the Devil (2010)
10    Django Unchained (2012)                  39      6       The Lives of Others (2006)


RRF puts Inception first, because it ranks high in both lists. The score sum puts it only
fourth: the DiCaprio movies with a higher BM25 score come first, and the vector clause adds
less than 1 point, too little to change their order. RRF gives both rankings the same
weight, so movies with a strong plot match also move up. Compare the RRF column with the
results of FAISS, OpenSearch, and pgvector in ch06-04: Lucene's filtered vector search
returns the exact filtered ranking here, so its fused ranking resembles the pgvector one.

## Quantized vectors

For lever 2, Lucene offers vector formats that store each component with fewer bits: 7 or
4 bits with scalar quantization, or a single bit with binary quantization. The HNSW graph
search then compares the compact codes. The format is chosen per field through the codec,
and the full vectors stay in the index, so the application can rescore the best candidates
with them. We build one index per format and measure recall@10 against exact search over
all five queries in two ways: the top 10 of the quantized search, and the top 10 after
rescoring its best 50 candidates with the full vectors.

In [11]:
import org.apache.lucene.codecs.KnnVectorsFormat;
import org.apache.lucene.codecs.lucene99.Lucene99HnswVectorsFormat;
import org.apache.lucene.codecs.lucene99.Lucene99HnswScalarQuantizedVectorsFormat;
import org.apache.lucene.codecs.lucene102.Lucene102HnswBinaryQuantizedVectorsFormat;
import org.apache.lucene.codecs.lucene101.Lucene101Codec;

IndexWriterConfig configFor(KnnVectorsFormat format) {
    return new IndexWriterConfig(new StandardAnalyzer()).setCodec(new Lucene101Codec() {
        @Override public KnnVectorsFormat getKnnVectorsFormatForField(String field) { return format; }
    });
}

// The full-precision vector of a document, read back from the index
float[] fullVector(IndexReader r, int doc) throws IOException {
    LeafReaderContext leaf = r.leaves().get(ReaderUtil.subIndex(doc, r.leaves()));
    FloatVectorValues values = leaf.reader().getFloatVectorValues("embedding");
    KnnVectorValues.DocIndexIterator it = values.iterator();
    it.advance(doc - leaf.docBase);
    return values.vectorValue(it.index()).clone();
}

Map<String, KnnVectorsFormat> formats = new LinkedHashMap<>();
formats.put("float32", new Lucene99HnswVectorsFormat());
formats.put("7-bit scalar", new Lucene99HnswScalarQuantizedVectorsFormat(16, 100, 1, 7, false, null, null));
formats.put("4-bit scalar", new Lucene99HnswScalarQuantizedVectorsFormat(16, 100, 1, 4, true, null, null));
formats.put("1-bit binary", new Lucene102HnswBinaryQuantizedVectorsFormat());

System.out.printf("%-14s %-8s %-18s %s%n", "Format", "Size", "Recall@10, top 10", "Recall@10, top 50 rescored");
for (var entry : formats.entrySet()) {
    Directory dir = buildIndex(configFor(entry.getValue()));
    try (IndexReader r = DirectoryReader.open(dir)) {
        IndexSearcher s = new IndexSearcher(r);
        double plain = 0, rescored = 0;
        for (float[] v : QUERIES.values()) {
            List<Integer> reference = exactRanking(v, m -> true);
            ScoreDoc[] top50 = s.search(new KnnFloatVectorQuery("embedding", v, 50), 50).scoreDocs;
            List<Integer> found = new ArrayList<>();
            Map<Integer, Float> exactScore = new HashMap<>();
            for (ScoreDoc sd : top50) {
                int id = s.storedFields().document(sd.doc).getField("id").numericValue().intValue();
                found.add(id);
                exactScore.put(id, dot(fullVector(r, sd.doc), v));
            }
            plain += overlap(found, reference, 10) / 10.0;
            List<Integer> reordered = found.stream()
                .sorted(Comparator.comparingDouble((Integer i) -> -exactScore.get(i))).toList();
            rescored += overlap(reordered, reference, 10) / 10.0;
        }
        System.out.printf("%-14s %-8s %-18.2f %.2f%n", entry.getKey(),
            String.format("%.0f MB", bytesOnDisk(dir) / 1e6), plain / QUERIES.size(), rescored / QUERIES.size());
    }
}

Format         Size     Recall@10, top 10  Recall@10, top 50 rescored
float32        42 MB    1.00               1.00
7-bit scalar   52 MB    0.98               1.00
4-bit scalar   47 MB    0.94               1.00
1-bit binary   44 MB    0.68               0.98


The 7-bit and 4-bit codes are almost as good as the floats. The 1-bit codes lose many of the
exact neighbors in their own top 10, but most of them are among the best 50 candidates, and
rescoring with the full vectors brings them back. The index on disk does not shrink: the
codes are stored next to the full vectors. What shrinks is the data the graph search touches
in memory, which is 4, 8, or 32 times smaller than the floats. Lucene 10.2 leaves the
rescoring to the application, as in this cell.

## Summary

| Book section 6.5 | In Lucene |
|---|---|
| Vector field next to text and numeric fields | `KnnFloatVectorField` with `VectorSimilarityFunction.COSINE` |
| Approximate k-NN | `KnnFloatVectorQuery`: one HNSW graph per segment; score $(1 + \cos)/2$ |
| Filtering during the search | filter as the fourth argument of `KnnFloatVectorQuery` |
| Post-filtering pitfall | unfiltered vector query plus a `FILTER` clause returns too few results |
| Hybrid search | Boolean query: the BM25 clauses dominate the score sum |
| Rank fusion | not built in: run both queries, fuse in the application |
| Fewer bits per vector | 7-, 4-, or 1-bit vector formats through the codec; rescore with the full vectors |

## Try it yourself

**1. Another query.** Replace `q` with another precomputed query, for example
`QUERIES.get("a shark terrorizes a beach town")`, and rerun the k-NN cells. Which movies does
the vector part find?

**2. Boost the vector clause.** Wrap the vector query in `new BoostQuery(knn50, 10f)` before
adding it to the hybrid query. How large must the boost be before the dense part changes the
order of the top 10?

In [12]:
Query boosted = new BooleanQuery.Builder()
    .add(new BoostQuery(knn50, 10f), BooleanClause.Occur.SHOULD)
    .add(actor, BooleanClause.Occur.SHOULD)
    .add(genre, BooleanClause.Occur.SHOULD)
    .add(filter, BooleanClause.Occur.FILTER)
    .build();
for (ScoreDoc sd : searcher.search(boosted, 5).scoreDocs)
    System.out.printf("%-40s %.4f%n", title(movieId(sd.doc)), sd.score);

Inception (2010)                         12.3454
The Departed (2006)                      12.2848
Shutter Island (2010)                    12.1841
The Wolf of Wall Street (2013)           12.0097
Django Unchained (2012)                  11.3431


**3. A milder filter.** Change the year in the range query to 1990 and drop the rating
condition. Does the post-filtered Boolean query still lose results?